# Data Quality Investigation: Newcastle Library Loans

This notebook applies steps 1–4 of Roy Ruddle’s six-step data quality workflow
to the Newcastle Libraries Loans dataset from Data Mill North.

Dataset URL: https://datamillnorth.org/dataset/newcastle-libraries-loans-e6qw9

Topic: Leisure and Tourism

Note from metadata:
"Blank means no data available."

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import datetime

from vizdataquality import calculate as vdqc, datasets as vdqd, plot as vdqp, report as vdqr


: 

## Loading the dataset

In [ ]:
dataset_filename = "../data/newcastle_loans.csv"
df1 = pd.read_csv(dataset_filename)
num_rows = df1.shape[0]
num_col = df1.shape[1]
print("rows:", num_rows, "|| columns:", num_col)
print(df1.columns)
df_output1 = vdqc.calc(df1) # creating a dataframe of data quality metrics for each column
print(df_output1)


## Initialising the report

In [ ]:
overwrite = True
report_folder = '../reports' # The folder in which the output is stored (it must exist).

if True:
    report_filename = 'newcastle_report.html'
    table_kw = {}
else:
    report_filename = 'newcastle_report.tex'
    mpl.rcParams['savefig.dpi'] = 300 # Save figures at 300 dpi
    table_kw={'position': 'h!'} # Ask Latex to place each image 'exactly here'

report = vdqr.Report()

report.add_title('Data quality issues and profiling of a Nwecastle library loan dataset')

text = 'This report has been produced by applying the vizdataquality package to investigate data quality of a dataset about Monthly loan figures of Newcastle libraries.'
text = text + ' The data is (c) Newcastle City Council, https://datamillnorth.org/dataset/newcastle-libraries-loans-e6qw9 ..'
report.add_heading('Introduction', text=text)

text = 'The first 4 workflow steps investigate data quality and clean it in a structured manner.'
report.paragraph(text)

# Workflow step 1: Looking at your data

In [ ]:
cols = ['Number of missing values']
missing_sorted = df_output1[cols].sort_values('Number of missing values', ascending=False).head(20)

fig_kw = {'size_inches': (10, 4), 'constrained_layout': True}
vdqp.plotgrid('scalars', missing_sorted, num_cols=1, vert=False, fig_kw=fig_kw)

cols = ['Data type', 'Example value', 'Number of values', 'Number of missing values', 'Uniqueness', 'Data type conflict']
summary = df_output1[cols].copy()

all_missing = summary[summary['Number of values'] == 0]
sample_normal = summary[summary['Number of values'] > 0].head(10)

display_df = pd.concat([all_missing.head(10), sample_normal])  # keeping it short for readability
vdqp.table(display_df, include_index=True, ax_kw={'title': 'Selected variable summary (all-missing + sample normal)'}, loc='center') # TITLE CANT BE SEEN ??

missing_col = "Number of missing values"
missing_distribution = (
    df_output1[missing_col]
    .value_counts()
    .sort_index()
    .rename_axis("Number of missing values per column")
    .reset_index(name="Number of columns")
)
missing_distribution
vdqp.table(missing_distribution, include_index=False, loc='center')



## Cleaning the table

In [ ]:
# Identify columns that are entirely missing

missing_fraction = df1.isna().mean()      # fraction missing per column (0.0 → 1.0)
structural_cols = missing_fraction[missing_fraction == 1.0].index.tolist()

print("Structural all-missing columns:", structural_cols[:10], "..." if len(structural_cols)>10 else "")
print("Count:", len(structural_cols))

# Remove all-missing columns
df2 = df1.drop(columns=structural_cols)
df_output2 = vdqc.calc(df2)

print("Before:", df1.shape)
print("After:", df2.shape)

df_output2 = vdqc.calc(df2)

# Adding to the report
report.step1(
    f"The dataset is in a wide table format (rows=libraries, columns=months). "
    f"I detected {len(structural_cols)} columns with 100% missingness (missing fraction = 1.0), which correspond to trailing time periods."
    f"These are treated as structural placeholders for profiling and removed before Steps 2–4 to prevent distorted summaries and unreadable plots. "
)

## Observations from step 1

* Dataset shape: 19 rows (libraries) × 205 columns (monthly values from 2008 onwards).

* Structure is has a wide table format: one entity per row, one time period per column.

* Columns from 2024-9 onwards contain contain 100% NaN (missing) values.

* These trailing columns are classified as float64 because they only contain NaN values.

* The Library column has 0 missing values and are all unique → stable entity identifier.

* No data type conflicts detected.

* Columns with 100% missing values will be removed before proceeding to later workflow steps.

* Plotting all 205 columns was unreadable. This showed me that summarisation strategies are necessary for wide datasets.

---

### Missingness Distribution Insight

The distribution of missing counts shows:

50 columns with 0 missing values (fully complete)

148 columns with moderate partial missingness

7 columns with 19 missing values (fully empty)

This pattern indicates clustered structural missingness, rather than uniformly random missing values.

---

## Design implications from step 1

* Step 1 should include an explicit structural detection rule (e.g., missing fraction = 1.0).

* Missingness should be summarised at the dataset level (distribution), not only via top-N plots.

* Wide-format datasets require layout-aware reporting (top-N plots + distribution summary).

* Structural placeholders should optionally be excluded before deeper profiling steps.

* Future testing across other dataset types will determine whether this structural pattern generalises.

# Workflow step 2: Watch out for special values

In [ ]:
# Look for common sentinel values in numeric columns
numeric_cols = df2.select_dtypes(include='number').columns

for col in numeric_cols:
    if df2[col].isin([-999, -1, 999, 9999]).any():
        print("Potential sentinel value in:", col)

df2.select_dtypes(include='object').apply(lambda x: x.unique())


# Identify object columns
object_cols = df2.select_dtypes(include='object').columns

# Check text sentinel values
common_text_sentinels = {"n/a", "unknown", "-", "null", "none", ""}

for col in object_cols:
    lower_values = df2[col].astype(str).str.strip().str.lower()
    if lower_values.isin(common_text_sentinels).any():
        print("Potential text sentinel in:", col)


## Checking the potential sentinel value:

In [ ]:
sentinels = [-999, -1, 999, 9999]

col = "2011-05"
df2.loc[df2[col].isin(sentinels), ["Library", col]]


In [ ]:
report.step2(
    "Numeric variables were scanned for common sentinel encodings (e.g., -999, -1, 999, 9999)."
    "One occurrence of value 999 was detected (Cruddas Park, 2011-05), but inspection confirmed this represents a plausible loan count rather than a placeholder encoding. "
    "Text variables were checked case-insensitively for common placeholder tokens (e.g., 'n/a', 'unknown', 'null'), and none were found. "
    "Missing values appear to be consistently encoded as NaN. "
    "No recoding was required before proceeding."
)

## Observations from step 2

* One occurrence of value 999 was detected in column 2011-05.

* Manual inspection confirmed this represents a plausible loan count (Cruddas Park library) rather than a placeholder encoding.

* No other numeric or text-based sentinel values were identified.

* Missing values appear to be consistently encoded using NaN.

---

## Design Implications (Step 2)

* Automated sentinel detection should flag suspicious numeric tokens but may need manual contextual validation.

* Sentinel detection must distinguish between legitimate domain values and placeholder encodings.

* The pipeline should treat numeric sentinel detection as a warning step, not an automatic deletion rule.

* For this dataset, no recoding was required before proceeding to Step 3.

The presence of a value (999) that could potentially be interpreted as a sentinel highlights the importance of domain-aware validation rather than rigid threshold-based cleaning.

# Workflow step 3: Is any data missing?

In [ ]:
report.step3(
    "Following the removal of structural all-missing columns in Step 1 and verification of sentinel encodings in Step 2, no remaining variables contain missing values. "
    "All monthly loan counts are complete for the retained time periods. "
    "The dataset is therefore suitable for variable-level validation without further imputation."
)

## Observations (Step 3)

* After removal of structural all-missing columns in Step 1, no remaining variables contain missing values.

---

## Design Implications (Step 3)

Pipeline should separate:

* Structural missingness (Step 1)

* Encoded missingness (Step 2)

* Residual missingness (Step 3)

# Workflow step 4: Check each variable (is it what you expect?)
This step focuses on variable-level validation, examining whether each column behaves consistently with expectations, in line with the structured profiling workflow proposed by Ruddle et al. (2024).

In [ ]:
# Copying the dataframe
df3 = df2.copy()

# Checking the uniqueness of library names
unique_lib = df3['Library'].nunique()
duplicates_lib = df3['Library'].duplicated().sum()

print("Unique library names:", unique_lib)
print("Duplicate library names:", duplicates_lib)

# Checking for negative values in numeric columns
numeric_cols = df3.select_dtypes(include='number').columns
negative_values = (df3[numeric_cols] < 0).sum().sum() #### WHYYYYYY????
print("Negative values in numeric columns:", negative_values)

# Checking max values for any spikes
summary = df3[numeric_cols].describe().T
top_by_max = summary.sort_values('max', ascending=False).head(10)[['min','mean','max']]
print(top_by_max)

# Checking min values for any anomalies
summary = df3[numeric_cols].describe().T
top_by_min = summary.sort_values('min', ascending=True).head(10)[['min','mean','max']]
print(top_by_min)

# Going through every column and keeping the column if it has exactly 1 unique value (i.e., constant across all libraries)
constant_cols = [col for col in numeric_cols if df3[col].nunique() == 1]
print("Columns with identical values across all libraries:", constant_cols)

# Checking for non-integer values in numeric columns
vals = df3[numeric_cols]

non_int_mask = vals.notna() & (vals % 1 != 0) # Ignoring missing values

non_integer_values = int(non_int_mask.sum().sum())
print("Non-integer numeric values:", non_integer_values)

### Low-Mean Month Analysis

In [ ]:
monthly_means = df3[numeric_cols].mean().sort_values()
monthly_means.head(10)

In [ ]:
report.step4(
    "Library identifiers were confirmed to be unique, with no duplicate names detected. "
    "Monthly loan variables contain no negative or non-integer values. "
    "Loan counts range from 0 to 56,618 across the dataset, with peak activity occurring between 2009–2010. "
    "Several months in 2020 exhibit substantially reduced average loan volumes, consistent with temporary operational disruptions rather than data error. "
    "No implausible extreme values or constant columns were identified."
)

## Observations from step 4:

* Loan counts range from 0 to 56,618 across the dataset.

* The highest monthly maxima occur between 2009–2010, indicating peak loan activity during that period.

* Several months in 2020 exhibit extremely low mean loan counts (e.g., April–June 2020), with some libraries reporting zero loans.

* Isolated zero values appear in earlier years (e.g., 2011-01, 2014-04), suggesting localised variation rather than systemic error.

* All numeric values are whole numbers (stored as floats due to missing data representation).

## Design implications from step 4:

* Variable-level checks must include plausibility validation (e.g., negative values, constant columns, extreme spikes).

* Temporal datasets may exhibit real-world structural shifts (e.g., significant drops in usage), which should not be misclassified as data errors.

Reporting should distinguish between:

* structural anomalies (e.g., all-missing columns),

* operational variation (e.g., temporary closures),

* and implausible values (e.g., negative counts).

The pipeline should:

* flag negative or non-integer values,

* highlight unusually low/high monthly means, but might need manaual removal because they might reflect real-world events.

# Generating the report

In [ ]:
report.save(os.path.join(report_folder, report_filename), overwrite=overwrite)